## Get Imports and Load Data

In [1]:
# Imports
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler # To scale BMI
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, r2_score
import joblib # For saving the model

# Load cleaned data:
df = pd.read_csv("data/processed/cleaned_normalized.csv")

## Load Cleaned Data and Split:

In [2]:
# Load full "demo" data to get BMI & ethnicity
demo_full = pd.read_csv("data/processed/demo_clean.csv")

# Merge back to get BMI and ethnicity
df = df.merge(demo_full[["SEQN", "BMXBMI", "RIDRETH1"]], on="SEQN")

# Scale BMI
scaler = StandardScaler()
df["BMXBMI_scaled"] = scaler.fit_transform(df[["BMXBMI"]])

# Enccode ethnicity (RIDRETH1: 1-6)
df = pd.get_dummies(df, columns=["RIDRETH1"], PREFIX="eth", drop_first=True)

KeyError: "['BMXBMI'] not in index"

## Update Features & Train Model

In [ ]:
# Features (X) and target (Y)
X = df[["RIDAGEYR_scaled", "RIAGENDR", "BMXBMI_scaled"]] + [col for col in df.columns if col.startswith("eth_")]]
y = df["BPXOSY1_scaled"] # Systolic BP (target)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Model
model = LinearRegression()
model.fit(X_train, y_train)

## Predict and Evaluate

In [ ]:
y_pred = model.predict(X_test)

print("RMSE:", root_mean_squared_error(y_test, y_pred))
print("R²:", r2_score(y_test, y_pred))

## Save Model

In [ ]:
os.makedirs("models", exist_ok=True)
joblib.dump(model, "models/bp_model_improved.pkl")